In [7]:
# Import needed files
from pynq import Overlay, allocate
import numpy as np
import time
from pathlib import Path

print("Imports successful")

Imports successful


In [8]:
# Check to see if vivado inport is working
try: 
    bnn_overlay = Overlay('/home/xilinx/jupyter_notebooks/MNIST/vivado/bnn_top.bit')
    print("Bitstream loaded successfully!")
    print(f"\nOverlay contains: {bnn_overlay.ip_dict.keys()}")
except Exception as e:
    print(f"Error loading bitstream: {e}")
# DMA and IP Setup
try:
    # Get BNN IP
    bnn_ip = bnn_overlay.bnn_top_0
    print(f"   BNN IP found: {bnn_ip}")
    # Get DMA
    dma = bnn_overlay.axi_dma
    dma_send = dma.sendchannel
    dma_recv = dma.recvchannel
    print(f"      DMA found: {dma}")
    print(f"   Send channel: {dma_send}")
    print(f"Receive channel: {dma_recv}")
except AttributeError as e:
    print(f"✗ Error accessing IP blocks: {e}")
    print("\nAvailable IPs:", list(bnn_overlay.ip_dict.keys()))
    print("\nMake sure your Vivado block design includes:")
    print("  - bnn_top_0 (your HLS IP)")
    print("  - axi_dma_0 (AXI DMA)")
    raise

Bitstream loaded successfully!

Overlay contains: dict_keys(['bnn_top_0', 'axi_dma', 'zynq_ultra_ps_e_0'])
   BNN IP found: <pynq.overlay.DefaultIP object at 0xffff56cb5c90>
      DMA found: <pynq.lib.dma.DMA object at 0xffff57e36650>
   Send channel: <pynq.lib.dma._SDMAChannel object at 0xffff5728bd60>
Receive channel: <pynq.lib.dma._SDMAChannel object at 0xffff5728be20>


In [15]:
# Load weights into PL BRAM
WEIGHT_DIR = "/home/xilinx/jupyter_notebooks/MNIST/weights"

# Register offets from xbnn_top_hw.h
AP_CTRL            = 0x00
L1_WEIGHTS_ADDR    = 0x10   # l1_weights[31:0]  (gmem0 phys addr, low)
L1_WEIGHTS_ADDR_HI = 0x14   # l1_weights[63:32] (gmem0 phys addr, high)
L2_WEIGHTS_ADDR    = 0x1C   # l2_weights[31:0]  (gmem1 phys addr, low)
L2_WEIGHTS_ADDR_HI = 0x20   # l2_weights[63:32] (gmem1 phys addr, high)
L3_WEIGHTS_ADDR    = 0x28   # l3_weights[31:0]  (gmem2 phys addr, low)
L3_WEIGHTS_ADDR_HI = 0x2C   # l3_weights[63:32] (gmem2 phys addr, high)
LOAD_WEIGHTS_OFF   = 0x34   # load_weights[0], bit 0

INPUT_STREAM_WORDS = 25     # ceil(784 / 32)

def write_64bit_reg(ip, lo_off, hi_off, value):
    ip.write(lo_off, value & 0xFFFFFFFF)
    ip.write(hi_off, (value >> 32) & 0xFFFFFFFF)

def pack_weights_to_bytes_aligned(weight_array, row_width_bits, bus_width_bits):
    """Pack weights with alignment padding for m_axi bus width."""
    n_neurons = weight_array.shape[0]
    row_bytes = (row_width_bits + 7) // 8            # 98 for 784 bits
    aligned_bytes = (bus_width_bits + 7) // 8         # 128 for 1024 bits
    # Use the larger of the two for stride
    stride = max(row_bytes, aligned_bytes)
    
    buf = np.zeros(n_neurons * stride, dtype=np.uint8)
    for n in range(n_neurons):
        for i in range(row_width_bits):
            if weight_array[n][i] < 0:
                byte_idx = n * stride + (i // 8)
                bit_idx  = i % 8
                buf[byte_idx] |= (1 << bit_idx)
    return buf

# L1: 784-bit rows, 1024-bit bus (from synthesis: 784 -> 1024)
l1_packed = pack_weights_to_bytes_aligned(l1_w, 784, 1024)
# L2: 256-bit rows, 256-bit bus (from synthesis: 256 -> 256, no widening)
l2_packed = pack_weights_to_bytes_aligned(l2_w, 256, 256)
# L3: 256-bit rows, 256-bit bus (from synthesis: 256 -> 256, no widening)
l3_packed = pack_weights_to_bytes_aligned(l3_w, 256, 256)

print(f"L1: {len(l1_packed)} bytes (was 25088, now {256*128})")
print(f"L2: {len(l2_packed)} bytes")
print(f"L3: {len(l3_packed)} bytes")

# Load overlay
bnn_overlay = Overlay('/home/xilinx/jupyter_notebooks/MNIST/vivado/bnn_top.bit')
bnn_ip = bnn_overlay.bnn_top_0
dma = bnn_overlay.axi_dma

# Load weights FRESH
l1_w = np.load(f"{WEIGHT_DIR}/fc1_weights.npy")
l2_w = np.load(f"{WEIGHT_DIR}/fc2_weights.npy")
l3_w = np.load(f"{WEIGHT_DIR}/fc3_weights.npy")

l1_buf = allocate(shape=(len(l1_packed),), dtype=np.uint8)
l2_buf = allocate(shape=(len(l2_packed),), dtype=np.uint8)
l3_buf = allocate(shape=(len(l3_packed),), dtype=np.uint8)
np.copyto(l1_buf, l1_packed)
np.copyto(l2_buf, l2_packed)
np.copyto(l3_buf, l3_packed)

print(f"L1: {len(l1_packed)} bytes at 0x{l1_buf.physical_address:08x}")
print(f"L2: {len(l2_packed)} bytes at 0x{l2_buf.physical_address:08x}")
print(f"L3: {len(l3_packed)} bytes at 0x{l3_buf.physical_address:08x}")

# Load weights into BRAM
bnn_ip.write(LOAD_WEIGHTS_OFF, 1)
write_64bit_reg(bnn_ip, L1_WEIGHTS_ADDR, L1_WEIGHTS_ADDR_HI, l1_buf.physical_address)
write_64bit_reg(bnn_ip, L2_WEIGHTS_ADDR, L2_WEIGHTS_ADDR_HI, l2_buf.physical_address)
write_64bit_reg(bnn_ip, L3_WEIGHTS_ADDR, L3_WEIGHTS_ADDR_HI, l3_buf.physical_address)
bnn_ip.write(AP_CTRL, 0x01)
while (bnn_ip.read(AP_CTRL) & 0x02) == 0:
    pass
print("Weights loaded.")

# Refresh DMA without reprogramming
bnn_overlay = Overlay('/home/xilinx/jupyter_notebooks/MNIST/vivado/bnn_top.bit', download=False)
bnn_ip = bnn_overlay.bnn_top_0
dma = bnn_overlay.axi_dma
dma_send = dma.sendchannel
dma_recv = dma.recvchannel
print(f"Send running: {dma_send.running}")
print(f"Recv running: {dma_recv.running}")

# Inference function
def run_inference(image_flat):
    input_buf  = allocate(shape=(INPUT_STREAM_WORDS,), dtype=np.uint32)
    output_buf = allocate(shape=(10,), dtype=np.int16)

    for w in range(INPUT_STREAM_WORDS):
        val = 0
        for b in range(32):
            idx = w * 32 + b
            if idx < 784 and image_flat[idx] == 0:
                val |= (1 << b)
        input_buf[w] = val

    bnn_ip.write(LOAD_WEIGHTS_OFF, 0)
    dma_recv.transfer(output_buf)
    bnn_ip.write(AP_CTRL, 0x01)
    dma_send.transfer(input_buf)

    dma_send.wait()
    dma_recv.wait()

    scores = output_buf.copy()
    predicted = int(np.argmax(scores))

    del input_buf, output_buf
    return predicted, scores


test_image = np.ones(784, dtype=np.uint8)
predicted, scores = run_inference(test_image)

expected = [6, -88, 14, 12, -104, -12, 8, -30, 98, 58]
print(f"Expected: {expected}  -> class 8")
print(f"Got:      {list(scores)}  -> class {predicted}")

if list(scores) == expected and predicted == 8:
    print("\TEST PASSED")
else:
    print("\nFAILED")


L1: 32768 bytes (was 25088, now 32768)
L2: 8192 bytes
L3: 320 bytes
L1: 32768 bytes at 0x780b0000
L2: 8192 bytes at 0x7808c000
L3: 320 bytes at 0x7809c000
Weights loaded.
Send running: True
Recv running: True
Expected: [6, -88, 14, 12, -104, -12, 8, -30, 98, 58]  -> class 8
Got:      [6, -88, 14, 12, -104, -12, 8, -30, 98, 58]  -> class 8

GOLDEN TEST PASSED


In [16]:
# HW Implementation setup
def fpga_hardware_testbench(test_inputs, test_labels):
    """
    Processes all MNIST test images one at a time.
    Measures timing, accuracy, and throughput.

    Args:
        test_inputs: (N, 784) uint8 array, values 0 or 1
        test_labels: (N,) int array, values 0-9

    Returns:
        results: Dictionary with all metrics
    """

    N = len(test_inputs)

    # Warm-up: run a few inferences to stabilize DMA/cache behavior
    dummy_img = np.ones(784, dtype=np.uint8)
    for _ in range(10):
        run_inference(dummy_img)
    print("Warm-up complete")

    # TESTBENCH
    print(f"\nProcessing {N:,} test images...")
    print("-" * 70)

    correct = 0
    total = 0
    individual_times = []

    start_total = time.perf_counter()

    for i in range(N):
        start_img = time.perf_counter()
        pred, _ = run_inference(test_inputs[i])
        end_img = time.perf_counter()

        individual_times.append(end_img - start_img)
        total += 1
        if pred == test_labels[i]:
            correct += 1

        # Progress update (matches batch reporting cadence)
        if (i + 1) % 1000 == 0:
            elapsed = time.perf_counter() - start_total
            print(f"  [{i+1:>5}/{N}]  acc so far: {100*correct/(i+1):.2f}%  "
                  f"elapsed: {elapsed:.1f}s")

    end_total = time.perf_counter()

    # METRICS
    total_time = end_total - start_total
    accuracy = 100.0 * correct / total
    avg_time_per_image_ms = (total_time / total) * 1000
    throughput_images_per_sec = total / total_time

    individual_times = np.array(individual_times)
    avg_img_time_ms = np.mean(individual_times) * 1000
    min_img_time_ms = np.min(individual_times) * 1000
    max_img_time_ms = np.max(individual_times) * 1000

    # Computational metrics (same op count as SW baseline)
    ops_fc1 = 784 * 256
    ops_fc2 = 256 * 256
    ops_fc3 = 256 * 10
    total_ops_per_inference = ops_fc1 + ops_fc2 + ops_fc3  # 268,800
    total_ops_executed = total_ops_per_inference * total
    ops_per_second = total_ops_per_inference * throughput_images_per_sec
    gops_per_second = ops_per_second / 1e9

    print("\n" + "=" * 70)
    print("RESULTS")
    print("=" * 70)

    print("\n1. ACCURACY RESULTS:")
    print(f"   Total images processed:  {total:,}")
    print(f"   Correct predictions:     {correct:,}")
    print(f"   Incorrect predictions:   {total - correct:,}")
    print(f"   Accuracy:                {accuracy:.2f}%")

    print("\n2. TIMING RESULTS:")
    print(f"   Total execution time:    {total_time:.4f} seconds")
    print(f"   Average time per image:  {avg_time_per_image_ms:.4f} ms")
    print(f"   Throughput:              {throughput_images_per_sec:.2f} images/second")

    print("\n3. PER-IMAGE TIMING STATISTICS:")
    print(f"   Total images:            {total:,}")
    print(f"   Average image time:      {avg_img_time_ms:.4f} ms")
    print(f"   Min image time:          {min_img_time_ms:.4f} ms")
    print(f"   Max image time:          {max_img_time_ms:.4f} ms")

    print("\n4. COMPUTATIONAL ANALYSIS:")
    print(f"   Operations per image:    {total_ops_per_inference:,}")
    print(f"   Total operations:        {total_ops_executed:,}")
    print(f"   Operations per second:   {ops_per_second:.2e}")
    print(f"   Throughput:              {gops_per_second:.4f} GOPS")

    results = {
        'accuracy': {
            'total_images': total,
            'correct': correct,
            'incorrect': total - correct,
            'accuracy_percent': float(accuracy)
        },
        'timing': {
            'total_time_sec': float(total_time),
            'avg_time_per_image_ms': float(avg_time_per_image_ms),
            'throughput_images_per_sec': float(throughput_images_per_sec)
        },
        'per_image_stats': {
            'avg_time_ms': float(avg_img_time_ms),
            'min_time_ms': float(min_img_time_ms),
            'max_time_ms': float(max_img_time_ms)
        },
        'computational': {
            'ops_per_image': total_ops_per_inference,
            'total_ops_executed': total_ops_executed,
            'ops_per_second': float(ops_per_second),
            'gops_per_second': float(gops_per_second)
        }
    }

    return results


In [20]:
# Load binarized test data (exported from PC)
test_inputs = np.load(f"/home/xilinx/jupyter_notebooks/MNIST/database/test_inputs_binarized.npy")  # (10000, 784) uint8
test_labels = np.load(f"/home/xilinx/jupyter_notebooks/MNIST/database/test_labels_full.npy")       # (10000,) int64

print(f"Test data: {test_inputs.shape}, dtype={test_inputs.dtype}")
print(f"Labels:    {test_labels.shape}, dtype={test_labels.dtype}")
print(f"Unique input values: {np.unique(test_inputs)}")

# Run the testbench
fpga_results = fpga_hardware_testbench(test_inputs, test_labels)

Test data: (10000, 784), dtype=uint8
Labels:    (10000,), dtype=int64
Unique input values: [0 1]
Warm-up complete

Processing 10,000 test images...
----------------------------------------------------------------------
  [ 1000/10000]  acc so far: 19.00%  elapsed: 14.3s
  [ 2000/10000]  acc so far: 18.40%  elapsed: 28.5s
  [ 3000/10000]  acc so far: 17.73%  elapsed: 42.8s
  [ 4000/10000]  acc so far: 17.62%  elapsed: 57.1s
  [ 5000/10000]  acc so far: 17.36%  elapsed: 71.3s
  [ 6000/10000]  acc so far: 17.02%  elapsed: 85.5s
  [ 7000/10000]  acc so far: 16.73%  elapsed: 99.7s
  [ 8000/10000]  acc so far: 16.16%  elapsed: 113.9s
  [ 9000/10000]  acc so far: 16.00%  elapsed: 128.1s
  [10000/10000]  acc so far: 15.68%  elapsed: 142.4s

RESULTS

1. ACCURACY RESULTS:
   Total images processed:  10,000
   Correct predictions:     1,568
   Incorrect predictions:   8,432
   Accuracy:                15.68%

2. TIMING RESULTS:
   Total execution time:    142.3650 seconds
   Average time per imag

In [ ]:
# Compare results
sw_cpu_latency_ms  = 0.1209
sw_cpu_throughput   = 8269.89
sw_cpu_gops         = 2.2229

sw_gpu_latency_ms  = 0.1086
sw_gpu_throughput   = 9209.52
sw_gpu_gops         = 2.4755

fpga_latency_ms    = fpga_results['timing']['avg_time_per_image_ms']
fpga_throughput    = fpga_results['timing']['throughput_images_per_sec']
fpga_gops          = fpga_results['computational']['gops_per_second']
fpga_accuracy      = fpga_results['accuracy']['accuracy_percent']

print("=" * 70)
print("FPGA vs SOFTWARE BASELINE COMPARISON")
print("=" * 70)

print(f"\n{'Platform':<35} {'Latency(ms)':<14} {'Throughput':<16} {'GOPS':<10} {'Accuracy'}")
print("-" * 90)
print(f"{'PC CPU (PyTorch, x86)':<35} {sw_cpu_latency_ms:<14.4f} {sw_cpu_throughput:<16.2f} {sw_cpu_gops:<10.4f} {'98.50%'}")
print(f"{'PC GPU (PyTorch, RTX 5070 Ti)':<35} {sw_gpu_latency_ms:<14.4f} {sw_gpu_throughput:<16.2f} {sw_gpu_gops:<10.4f} {'98.50%'}")
print(f"{'FPGA (XCZU3EG, this design)':<35} {fpga_latency_ms:<14.4f} {fpga_throughput:<16.2f} {fpga_gops:<10.4f} {fpga_accuracy:.2f}%")

print(f"\nSpeedup vs PC CPU:  {sw_cpu_latency_ms / fpga_latency_ms:.2f}x")
print(f"Speedup vs PC GPU:  {sw_gpu_latency_ms / fpga_latency_ms:.2f}x")

print(f"\nNote: FPGA latency includes Python overhead + DMA transfers + kernel.")
print(f"      Kernel-only latency: ~0.97 us (322 cycles @ 300 MHz)")
print("=" * 70)

In [ ]:
# Free buffers
l1_buf.freebuffer()
l2_buf.freebuffer()
l3_buf.freebuffer()